In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
data_name = './datos_hkust_all.csv'

In [ ]:
data = pd.read_csv(data_name)

In [ ]:
data.head()

Indicadores:

**1. Nongraded problems**:
  - **n_attempted**: Number of problems attempted (at least once). Takes into account all the attempts.
  - **perc_attempted**: Percentaje of problems attempted (at least once). Counts every problem attempted as 1, eventhough the student has attempted it more than once.
  - **CFA**: Percentaje of problems correct on the first attempt (CFA). We consider a problem correct if its mark is 1; that is, if it is totally correct.
  - **avg_attempts**: Average number of attempts (taking into account only attempted exercises)
  - **avg_grade**: Average grade of problems attempted (taking account the maximum grade)
  - **n_show**: Number of times the student asks for problem_show

**2. Videos:**
  - **per_videos**: Percentaje of videos visualized
  - **n_pauses**: Number of times a student pauses the videos

**3. Forum:**
  - **n_threads**: Number of threads created
  - **n_responses**: Number of responses created
  - **n_comments**: Number of comments created

**4. Others:**
  - **dropped_out**: Abandono
  - **passed**: Curso superado
  - **n_days**: Number of different days a student access the course


In [ ]:
def indicadores_per_week(w,c,data,list_nongraded,days,videos):
  df = pd.read_csv('./Modelos/week{}.csv'.format(w),index_col=0)
  prob_list = list(list_nongraded['ID'].loc[list_nongraded['Chapter'].str.contains(str(c))])
  grs = pd.DataFrame(index = data['user_id'].unique(), columns=prob_list)

  ### Nongraded problems
  print('Problems')
  # Compute the list of grades per attempt of each student for each problem
  for i in prob_list:
    y = df.loc[(df['element_id'].str.contains(i))&(df['event_type']=='problem_check')]
    y = y.sort_values('attempts')
    users_id = list(y['user_id'])
    grades = list(y['grade'])
    max_grades = list(y['max_grade'])
    unique_users = np.unique(np.array(users_id))
    for u in unique_users:
      grs.loc[u,i] = []
    if len(users_id) == len(grades) == len(max_grades):
      for j in range(0,len(users_id)):
          # Per user, per problem: we have a list with all the grades in order of attempts
          grs.loc[users_id[j],i].append(grades[j]/max_grades[j])

  # All students
  grs = grs.fillna('not attempted')
  # Students who have answered to at least one problem
  grs_one = grs.loc[(grs!='not attempted').any(axis=1)]

  nongraded = pd.DataFrame(index = data['user_id'].unique(), columns=['List of grades first attempt','List of grades maximum grade','List of attempts'])
  nongraded['List of grades maximum grade'] = nongraded['List of grades maximum grade'].fillna({i: [] for i in nongraded.index})
  nongraded['List of grades first attempt'] = nongraded['List of grades first attempt'].fillna({i: [] for i in nongraded.index})
  users = grs_one.index

  for i in prob_list:
    for u in users:
      if type(grs_one.loc[u,i]) == list:
        if len(grs_one.loc[u,i]) >= 1:
          # Max grade to get average
          nongraded.loc[u,'List of grades maximum grade'].append(grs_one.loc[u,i][-1])
          # Grade of first attempt
          if grs_one.loc[u,i][0] == 1:
            nongraded.loc[u,'List of grades first attempt'].append(1)
          else:
            nongraded.loc[u,'List of grades first attempt'].append(0)

  indicadores = pd.DataFrame(index = data['user_id'].unique())
  # Number of problems attempted (at least once)
  indicadores['n_attempted'] = 0
  users = df['user_id'].loc[df['event_type']=='problem_check']
  for u in users:
    indicadores.loc[u,'n_attempted'] += 1
  # Percentaje of problems attempted (at least once)
  indicadores['per_attempted'] = [len(i)/len(prob_list) for i in nongraded['List of grades first attempt']]
  # Percentaje of problems correct on the first attempt (CFA)
  indicadores['CFA'] = [sum(i)/len(i) if len(i)!= 0 else '0 problems attempted' for i in nongraded['List of grades first attempt']]
  # Average grade of problems attempted (taking into account the maximum grade) -> ¿Tiene sentido si lo estamos haciendo por semanas?
  indicadores['avg_grade'] = [sum(i)/len(i) if len(i)!= 0 else '0 problems attempted' for i in nongraded['List of grades maximum grade']]

  # Average number of attempts (taking into account only attempted exercises)
  nongraded['List of attempts'] = nongraded['List of attempts'].fillna({i: [] for i in nongraded.index})
  for i in prob_list:
    y = df[(df['element_id'].str.contains(i)) & (df['event_type']=='problem_check')]
    y = y.sort_values('attempts')
    users_id = list(y['user_id'])
    attempts = list(y['attempts'])
    unique_users = np.unique(np.array(users_id))
    if len(users_id) == len(attempts):
        for u in unique_users:
          att = []
          for j in range(len(users_id)):
            if users_id[j] == u:
              att.append(attempts[j])
          nongraded['List of attempts'].loc[u].append(max(att))
  indicadores['avg_attempts'] = [sum(i)/len(i) if len(i)!= 0 else '0 problems attempted' for i in nongraded['List of attempts']]

  # Number of times the student asks for problem_show
  indicadores['n_show'] = 0
  y = df['user_id'].loc[(df['event_type']=='problem_show')]
  for user in y:
    indicadores.loc[user,'n_show'] += 1


  ### Videos
  print('Videos')
  # Percentaje of videos visualized
  vid_list = list(videos['ID'].loc[videos['Chapter'].str.contains(str(c))])
  play_video = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
  pause_stop_video = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
  for i in vid_list:
    play_video[i] = play_video[i].fillna({j: [] for j in play_video.index})
    pause_stop_video[i] = pause_stop_video[i].fillna({j: [] for j in pause_stop_video.index})
  for i in vid_list:
    # Play video
    users1 = list(df['user_id'].loc[(df['event_type']=='play_video') & (df['element_id'].str.contains(i))])
    seconds1 = list(df['old'].loc[(df['event_type']=='play_video') & (df['element_id'].str.contains(i))])
    for j in range(0,len(users1)):
      if not pd.isna(seconds1[j]):
        play_video.loc[users1[j],i].append(int(seconds1[j]))
    # Pause-stop video
    users2 = list(df['user_id'].loc[((df['event_type']=='stop_video') | (df['event_type']=='pause_video')) & (df['element_id'].str.contains(i))])
    seconds2 = list(df['old'].loc[((df['event_type']=='stop_video') | (df['event_type']=='pause_video')) & (df['element_id'].str.contains(i))])
    for j in range(0,len(users2)):
      if not pd.isna(seconds2[j]):
        pause_stop_video.loc[users2[j],i].append(int(seconds2[j]))
  videos_played = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
  videos_played = videos_played.T
  for i in vid_list:
    videos_played.loc[i] = list(videos['Seconds list'].loc[videos['ID']==i])
  videos_played = videos_played.T
  import ast
  for v in vid_list:
    for u in videos_played.index:
      play =  play_video.loc[u,v]
      pause = pause_stop_video.loc[u,v]
      if len(pause) > len(play):
        x = len(play)
        pause = pause[:len(play)]
      if len(play) > len(pause):
        x = len(pause)
        play = play[:len(pause)]
      if len(play) == len(pause):
        for i in range(0,len(play)):
          if pause[i] <= len(videos_played.loc[u,v]):
            if play[i] <= len(videos_played.loc[u,v]):
              if pause[i] > play[i]:
                videos_played.loc[u,v] = str(videos_played.loc[u,v])
                videos_played.loc[u,v] = ast.literal_eval(videos_played.loc[u,v].replace('.',','))
                if len(videos_played.loc[u,v][play[i]:pause[i]]) == pause[i]-play[i]:
                  videos_played.loc[u,v][play[i]:pause[i]] += np.ones(pause[i]-play[i])
  videos_percents = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
  for v in vid_list:
    videos_percents[v] = [np.sum(i)/len(i) for i in videos_played[v]]
  indicadores['per_videos'] = 0
  indicadores['per_videos'] = videos_percents.sum(axis=1)/len(vid_list)

  # Number of videos visualized (90% or more)
  indicadores['n_videos_watched'] = 0
  for u in indicadores.index:
    for v in videos_percents.columns:
      if videos_percents.loc[u,v] >= 0.9:
        indicadores.loc[u,'n_videos_watched'] += 1

  # Number of videos opened
  indicadores['n_videos_opened'] = 0
  print('maximum of: ', len(vid_list))
  for v in vid_list:
    vid_data = df['user_id'].loc[(df['element_id'].str.contains(v)) & (df['event_type'].str.contains('video'))]
    unique_users = vid_data.unique()
    for u in unique_users:
      indicadores.loc[u,'n_videos_opened'] += 1

  # Number of times the student pauses a video
  indicadores['n_pauses'] = 0
  users= df['user_id'].loc[(df['event_type']=='pause_video')]
  for u in users:
    indicadores.loc[u,'n_pauses'] += 1


  ### Forum
  print('Forum')
  # Number of threads created
  indicadores['n_threads'] = 0
  users = df['user_id'].loc[df['event_type']=='edx.forum.thread.created']
  for u in users:
    indicadores.loc[u,'n_threads'] += 1

  # Number of responses created
  indicadores['n_responses'] = 0
  users = df['user_id'].loc[df['event_type']=='edx.forum.response.created']
  for u in users:
    indicadores.loc[u,'n_responses'] += 1

  # Number of comments created
  indicadores['n_comments'] = 0
  users = df['user_id'].loc[df['event_type']=='edx.forum.comment.created']
  for u in users:
    indicadores.loc[u,'n_comments'] += 1


  ## Others
  print('Others')

  # Number of different days a student access the course
  indicadores['n_days'] = 0
  for d in days[w]:
    users_daily = df['user_id'].loc[df['time'].str.contains(d)]
    for u in set(list(users_daily)):
      indicadores.loc[u, 'n_days'] += 1

  indicadores.to_csv('./Modelos/indicadores_week{}.csv'.format(w))

  #return indicadores

In [ ]:
list_nongraded = pd.read_csv('./Indicadores/list_nongraded.csv')
list_nongraded

In [ ]:
videos = pd.read_csv('./Indicadores/videos.csv')
videos['Seconds list'] = [np.zeros((int(i))) for i in videos['Duration']]
videos

In [ ]:
import datetime
date_format = "%Y-%m-%d"
d1 = datetime.datetime.strptime("2015-04-28", date_format).date()
d2 = datetime.datetime.strptime("2015-06-30", date_format).date()
step = datetime.timedelta(days=7)
weeks = []
while d1 <= d2:
    weeks.append(d1.strftime(date_format))
    d1 += step

days = {}
date_format = "%Y-%m-%d"
for w in range(0,len(weeks[:-1])):
  days[w+1] = []
  d1 = datetime.datetime.strptime(weeks[w], date_format).date()
  d2 = datetime.datetime.strptime(weeks[w+1], date_format).date()
  step = datetime.timedelta(days=1)
  while d1 < d2:
      days[w+1].append(d1.strftime(date_format))
      d1 += step

In [ ]:
correspondence = {1:1, 2:1, 3:2, 4:3, 5:4, 6:5, 7: 5}
for w in correspondence.keys():
  print('Week:',w)
  print('Chapter:',correspondence[w])
  indicadores_per_week(w,correspondence[w],data,list_nongraded,days,videos)

In [ ]:
# Adding new indicators for the new configurations

In [ ]:
def indicadores_per_week_extra(w,c,data,days,videos):
  df = pd.read_csv('./Modelos/Weekly datasets/week{}.csv'.format(w),index_col=0)
  indicadores = pd.read_csv('./Modelos/Weekly indicadores/indicadores_week{}.csv'.format(w),index_col=0)

  ### Videos
  print('Videos')
  print('Computing percentaje of video visualized')
  # Percentaje of videos visualized
  vid_list = list(videos['ID'].loc[videos['Chapter'].str.contains(str(c))])
  play_video = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
  pause_stop_video = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
  for i in vid_list:
    play_video[i] = play_video[i].fillna({j: [] for j in play_video.index})
    pause_stop_video[i] = pause_stop_video[i].fillna({j: [] for j in pause_stop_video.index})

  print('\tComputing play_video and pause_stop_video')
  for i in vid_list:
    # Play video
    users1 = list(df['user_id'].loc[(df['event_type']=='play_video') & (df['element_id'].str.contains(i))])
    seconds1 = list(df['old'].loc[(df['event_type']=='play_video') & (df['element_id'].str.contains(i))])
    for j in range(0,len(users1)):
      if not pd.isna(seconds1[j]):
        play_video.loc[users1[j],i].append(int(seconds1[j]))
    # Pause-stop video
    users2 = list(df['user_id'].loc[((df['event_type']=='stop_video') | (df['event_type']=='pause_video')) & (df['element_id'].str.contains(i))])
    seconds2 = list(df['old'].loc[((df['event_type']=='stop_video') | (df['event_type']=='pause_video')) & (df['element_id'].str.contains(i))])
    for j in range(0,len(users2)):
      if not pd.isna(seconds2[j]):
        pause_stop_video.loc[users2[j],i].append(int(seconds2[j]))

  print('\tComputing videos_played')
  videos_played = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
  videos_played = videos_played.T
  for i in vid_list:
    videos_played.loc[i] = list(videos['Seconds list'].loc[videos['ID']==i])
  videos_played = videos_played.T
  import ast
  for v in vid_list:
    for u in videos_played.index:
      play =  play_video.loc[u,v]
      pause = pause_stop_video.loc[u,v]
      if len(pause) > len(play):
        x = len(play)
        pause = pause[:len(play)]
      if len(play) > len(pause):
        x = len(pause)
        play = play[:len(pause)]
      if len(play) == len(pause):
        for i in range(0,len(play)):
          if pause[i] <= len(videos_played.loc[u,v]):
            if play[i] <= len(videos_played.loc[u,v]):
              if pause[i] > play[i]:
                videos_played.loc[u,v] = str(videos_played.loc[u,v])
                videos_played.loc[u,v] = ast.literal_eval(videos_played.loc[u,v].replace('.',','))
                if len(videos_played.loc[u,v][play[i]:pause[i]]) == pause[i]-play[i]:
                  videos_played.loc[u,v][play[i]:pause[i]] += np.ones(pause[i]-play[i])

  print('\tComputing video_percents')
  videos_percents = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
  for v in vid_list:
    videos_percents[v] = [np.sum(i)/len(i) for i in videos_played[v]]

  print('Computing number of videos visualized')
  # Number of videos visualized (90% or more)
  indicadores['n_videos_watched'] = 0
  for u in indicadores.index:
    for v in videos_percents.columns:
      if videos_percents.loc[u,v] >= 0.9:
        indicadores.loc[u,'n_videos_watched'] += 1

  print('Computing number of videos opened')
  # Number of videos opened
  indicadores['n_videos_opened'] = 0
  print('maximum of: ', len(vid_list))
  for v in vid_list:
    vid_data = df['user_id'].loc[(df['element_id'].str.contains(v)) & (df['event_type'].str.contains('video'))]
    unique_users = vid_data.unique()
    for u in unique_users:
      indicadores.loc[u,'n_videos_opened'] += 1

  indicadores.to_csv('./Modelos/Weekly indicadores/Extra/indicadores_week{}.csv'.format(w))

In [ ]:
correspondence = {1:1, 2:1, 3:2, 4:3, 5:4, 6:5, 7: 5}
for w in correspondence.keys():
  print('\nWeek:',w)
  print('Chapter:',correspondence[w])
  indicadores_per_week_extra(w,correspondence[w],data,days,videos)

In [ ]:
# Indicadores all dataset
df = pd.read_csv('./Modelos/week1.csv',index_col=0)
for w in range(2,8):
  df = pd.concat([df, pd.read_csv('./Modelos/week{}.csv'.format(w),index_col=0)])
prob_list = list(list_nongraded['ID'])
grs = pd.DataFrame(index = data['user_id'].unique(), columns=prob_list)

### Nongraded problems
print('Problems')
# Compute the list of grades per attempt of each student for each problem
for i in prob_list:
  y = df.loc[(df['element_id'].str.contains(i))&(df['event_type']=='problem_check')]
  y = y.sort_values('attempts')
  users_id = list(y['user_id'])
  grades = list(y['grade'])
  max_grades = list(y['max_grade'])
  unique_users = np.unique(np.array(users_id))
  for u in unique_users:
    grs.loc[u,i] = []
  if len(users_id) == len(grades) == len(max_grades):
    for j in range(0,len(users_id)):
        # Per user, per problem: we have a list with all the grades in order of attempts
        grs.loc[users_id[j],i].append(grades[j]/max_grades[j])

# All students
grs = grs.fillna('not attempted')
# Students who have answered to at least one problem
grs_one = grs.loc[(grs!='not attempted').any(axis=1)]

nongraded = pd.DataFrame(index = data['user_id'].unique(), columns=['List of grades first attempt','List of grades maximum grade','List of attempts'])
nongraded['List of grades maximum grade'] = nongraded['List of grades maximum grade'].fillna({i: [] for i in nongraded.index})
nongraded['List of grades first attempt'] = nongraded['List of grades first attempt'].fillna({i: [] for i in nongraded.index})
users = grs_one.index

for i in prob_list:
  for u in users:
    if type(grs_one.loc[u,i]) == list:
      if len(grs_one.loc[u,i]) >= 1:
        # Max grade to get average
        nongraded.loc[u,'List of grades maximum grade'].append(grs_one.loc[u,i][-1])
        # Grade of first attempt
        if grs_one.loc[u,i][0] == 1:
          nongraded.loc[u,'List of grades first attempt'].append(1)
        else:
          nongraded.loc[u,'List of grades first attempt'].append(0)

indicadores = pd.DataFrame(index = data['user_id'].unique())
# Number of problems attempted (at least once)
indicadores['n_attempted'] = 0
users = df['user_id'].loc[df['event_type']=='problem_check']
for u in users:
  indicadores.loc[u,'n_attempted'] += 1
# Percentaje of problems attempted (at least once)
indicadores['per_attempted'] = [len(i)/len(prob_list) for i in nongraded['List of grades first attempt']]
# Percentaje of problems correct on the first attempt (CFA)
indicadores['CFA'] = [sum(i)/len(i) if len(i)!= 0 else '0 problems attempted' for i in nongraded['List of grades first attempt']]
# Average grade of problems attempted (taking into account the maximum grade) -> ¿Tiene sentido si lo estamos haciendo por semanas?
indicadores['avg_grade'] = [sum(i)/len(i) if len(i)!= 0 else '0 problems attempted' for i in nongraded['List of grades maximum grade']]

# Average number of attempts (taking into account only attempted exercises)
nongraded['List of attempts'] = nongraded['List of attempts'].fillna({i: [] for i in nongraded.index})

for i in prob_list:
  y = df[(df['element_id'].str.contains(i)) & (df['event_type']=='problem_check')]
  y = y.sort_values('attempts')
  users_id = list(y['user_id'])
  attempts = list(y['attempts'])
  unique_users = np.unique(np.array(users_id))
  if len(users_id) == len(attempts):
      for u in unique_users:
        att = []
        for j in range(len(users_id)):
          if users_id[j] == u:
            att.append(attempts[j])
        nongraded['List of attempts'].loc[u].append(max(att))

indicadores['avg_attempts'] = [sum(i)/len(i) if len(i)!= 0 else '0 problems attempted' for i in nongraded['List of attempts']]

# Number of times the student asks for problem_show
indicadores['n_show'] = 0
y = df['user_id'].loc[(df['event_type']=='problem_show')]
for user in y:
  indicadores.loc[user,'n_show'] += 1


### Videos
print('Videos')
# Percentaje of videos visualized
vid_list = list(videos['ID'])
play_video = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
pause_stop_video = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
for i in vid_list:
  play_video[i] = play_video[i].fillna({j: [] for j in play_video.index})
  pause_stop_video[i] = pause_stop_video[i].fillna({j: [] for j in pause_stop_video.index})
for i in vid_list:
  # Play video
  users1 = list(df['user_id'].loc[(df['event_type']=='play_video') & (df['element_id'].str.contains(i))])
  seconds1 = list(df['old'].loc[(df['event_type']=='play_video') & (df['element_id'].str.contains(i))])
  for j in range(0,len(users1)):
    if not pd.isna(seconds1[j]):
      play_video.loc[users1[j],i].append(int(seconds1[j]))
  # Pause-stop video
  users2 = list(df['user_id'].loc[((df['event_type']=='stop_video') | (df['event_type']=='pause_video')) & (df['element_id'].str.contains(i))])
  seconds2 = list(df['old'].loc[((df['event_type']=='stop_video') | (df['event_type']=='pause_video')) & (df['element_id'].str.contains(i))])
  for j in range(0,len(users2)):
    if not pd.isna(seconds2[j]):
      pause_stop_video.loc[users2[j],i].append(int(seconds2[j]))
videos_played = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
videos_played = videos_played.T
for i in vid_list:
  videos_played.loc[i] = list(videos['Seconds list'].loc[videos['ID']==i])
videos_played = videos_played.T
import ast
for v in vid_list:
  for u in videos_played.index:
    play =  play_video.loc[u,v]
    pause = pause_stop_video.loc[u,v]
    if len(pause) > len(play):
      x = len(play)
      pause = pause[:len(play)]
    if len(play) > len(pause):
      x = len(pause)
      play = play[:len(pause)]
    if len(play) == len(pause):
      for i in range(0,len(play)):
        if pause[i] <= len(videos_played.loc[u,v]):
          if play[i] <= len(videos_played.loc[u,v]):
            if pause[i] > play[i]:
              videos_played.loc[u,v] = str(videos_played.loc[u,v])
              videos_played.loc[u,v] = ast.literal_eval(videos_played.loc[u,v].replace('.',','))
              if len(videos_played.loc[u,v][play[i]:pause[i]]) == pause[i]-play[i]:
                videos_played.loc[u,v][play[i]:pause[i]] += np.ones(pause[i]-play[i])
videos_percents = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
for v in vid_list:
  videos_percents[v] = [np.sum(i)/len(i) for i in videos_played[v]]
indicadores['per_videos'] = 0
indicadores['per_videos'] = videos_percents.sum(axis=1)/len(vid_list)

# Number of times the student pauses a video
indicadores['n_pauses'] = 0
users= df['user_id'].loc[(df['event_type']=='pause_video')]
for u in users:
  indicadores.loc[u,'n_pauses'] += 1


### Forum
print('Forum')
# Number of threads created
indicadores['n_threads'] = 0
users = df['user_id'].loc[df['event_type']=='edx.forum.thread.created']
for u in users:
  indicadores.loc[u,'n_threads'] += 1

# Number of responses created
indicadores['n_responses'] = 0
users = df['user_id'].loc[df['event_type']=='edx.forum.response.created']
for u in users:
  indicadores.loc[u,'n_responses'] += 1

# Number of comments created
indicadores['n_comments'] = 0
users = df['user_id'].loc[df['event_type']=='edx.forum.comment.created']
for u in users:
  indicadores.loc[u,'n_comments'] += 1


## Others
print('Others')

# Number of different days a student access the course
indicadores['n_days'] = 0
for w in range(1,8):
  for d in days[w]:
    print(d)
    users_daily = df['user_id'].loc[df['time'].str.contains(d)]
    for u in set(list(users_daily)):
      indicadores.loc[u, 'n_days'] += 1

indicadores.to_csv('./Modelos/indicadores_28_4_14_06.csv')

In [ ]:
df

In [ ]:
# Adding extra indicators to all dataset
indicadores = pd.read_csv('./Modelos/indicadores_28_4_14_06.csv'.format(w),index_col=0)
indicadores

In [ ]:
### Videos
print('Videos')
print('Computing percentaje of video visualized')
# Percentaje of videos visualized
vid_list = list(videos['ID'])
play_video = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
pause_stop_video = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
for i in vid_list:
  play_video[i] = play_video[i].fillna({j: [] for j in play_video.index})
  pause_stop_video[i] = pause_stop_video[i].fillna({j: [] for j in pause_stop_video.index})

print('\tComputing play_video and pause_stop_video')
for i in vid_list:
  # Play video
  users1 = list(df['user_id'].loc[(df['event_type']=='play_video') & (df['element_id'].str.contains(i))])
  seconds1 = list(df['old'].loc[(df['event_type']=='play_video') & (df['element_id'].str.contains(i))])
  for j in range(0,len(users1)):
    if not pd.isna(seconds1[j]):
      play_video.loc[users1[j],i].append(int(seconds1[j]))
  # Pause-stop video
  users2 = list(df['user_id'].loc[((df['event_type']=='stop_video') | (df['event_type']=='pause_video')) & (df['element_id'].str.contains(i))])
  seconds2 = list(df['old'].loc[((df['event_type']=='stop_video') | (df['event_type']=='pause_video')) & (df['element_id'].str.contains(i))])
  for j in range(0,len(users2)):
    if not pd.isna(seconds2[j]):
      pause_stop_video.loc[users2[j],i].append(int(seconds2[j]))

print('\tComputing videos_played')
videos_played = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
videos_played = videos_played.T
for i in vid_list:
  videos_played.loc[i] = list(videos['Seconds list'].loc[videos['ID']==i])
videos_played = videos_played.T
import ast
for v in vid_list:
  for u in videos_played.index:
    play =  play_video.loc[u,v]
    pause = pause_stop_video.loc[u,v]
    if len(pause) > len(play):
      x = len(play)
      pause = pause[:len(play)]
    if len(play) > len(pause):
      x = len(pause)
      play = play[:len(pause)]
    if len(play) == len(pause):
      for i in range(0,len(play)):
        if pause[i] <= len(videos_played.loc[u,v]):
          if play[i] <= len(videos_played.loc[u,v]):
            if pause[i] > play[i]:
              videos_played.loc[u,v] = str(videos_played.loc[u,v])
              videos_played.loc[u,v] = ast.literal_eval(videos_played.loc[u,v].replace('.',','))
              if len(videos_played.loc[u,v][play[i]:pause[i]]) == pause[i]-play[i]:
                videos_played.loc[u,v][play[i]:pause[i]] += np.ones(pause[i]-play[i])

print('\tComputing video_percents')
videos_percents = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
for v in vid_list:
  videos_percents[v] = [np.sum(i)/len(i) for i in videos_played[v]]

print('Computing number of videos visualized')
# Number of videos visualized (90% or more)
indicadores['n_videos_watched'] = 0
for u in indicadores.index:
  for v in videos_percents.columns:
    if videos_percents.loc[u,v] >= 0.9:
      indicadores.loc[u,'n_videos_watched'] += 1

print('Computing number of videos opened')
# Number of videos opened
indicadores['n_videos_opened'] = 0
print('maximum of: ', len(vid_list))
for v in vid_list:
  vid_data = df['user_id'].loc[(df['element_id'].str.contains(v)) & (df['event_type'].str.contains('video'))]
  unique_users = vid_data.unique()
  for u in unique_users:
    indicadores.loc[u,'n_videos_opened'] += 1

#indicadores.to_csv('./Modelos/Weekly indicadores/Extra/indicadores_week{}.csv'.format(w))

In [ ]:
indicadores.to_csv('./Modelos/Weekly indicadores/Extra/indicadores_all_weeks.csv')

In [ ]:
indicadores

# **Cummulative datasets**

In [ ]:
def indicadores_per_week_cummulative(w,chaps,data,list_nongraded,days,videos):
  if w == 1:
    df = pd.read_csv('./Modelos/Cummulative/Weeks/weeks{}.csv'.format('1_1'),index_col=0)
    print('weeks{}.csv'.format('1_1'))
  else:
    df = pd.read_csv('./Modelos/Cummulative/Weeks/weeks{}.csv'.format(str(1)+'_'+str(w)),index_col=0)
    print('weeks{}.csv'.format(str(1)+'_'+str(w)))
  prob_list = []
  for c in chaps:
    [prob_list.append(i) for i in list_nongraded['ID'].loc[list_nongraded['Chapter'].str.contains(str(c))]]
  grs = pd.DataFrame(index = data['user_id'].unique(), columns=prob_list)

  ### Nongraded problems
  print('Problems')
  # Compute the list of grades per attempt of each student for each problem
  for i in prob_list:
    y = df.loc[(df['element_id'].str.contains(i))&(df['event_type']=='problem_check')]
    y = y.sort_values('attempts')
    users_id = list(y['user_id'])
    grades = list(y['grade'])
    max_grades = list(y['max_grade'])
    unique_users = np.unique(np.array(users_id))
    for u in unique_users:
      grs.loc[u,i] = []
    if len(users_id) == len(grades) == len(max_grades):
      for j in range(0,len(users_id)):
          # Per user, per problem: we have a list with all the grades in order of attempts
          grs.loc[users_id[j],i].append(grades[j]/max_grades[j])

  # All students
  grs = grs.fillna('not attempted')
  # Students who have answered to at least one problem
  grs_one = grs.loc[(grs!='not attempted').any(axis=1)]

  nongraded = pd.DataFrame(index = data['user_id'].unique(), columns=['List of grades first attempt','List of grades maximum grade','List of attempts'])
  nongraded['List of grades maximum grade'] = nongraded['List of grades maximum grade'].fillna({i: [] for i in nongraded.index})
  nongraded['List of grades first attempt'] = nongraded['List of grades first attempt'].fillna({i: [] for i in nongraded.index})
  users = grs_one.index

  for i in prob_list:
    for u in users:
      if type(grs_one.loc[u,i]) == list:
        if len(grs_one.loc[u,i]) >= 1:
          # Max grade to get average
          nongraded.loc[u,'List of grades maximum grade'].append(grs_one.loc[u,i][-1])
          # Grade of first attempt
          if grs_one.loc[u,i][0] == 1:
            nongraded.loc[u,'List of grades first attempt'].append(1)
          else:
            nongraded.loc[u,'List of grades first attempt'].append(0)

  indicadores = pd.DataFrame(index = data['user_id'].unique())
  # Number of problems attempted (at least once)
  indicadores['n_attempted'] = 0
  users = df['user_id'].loc[df['event_type']=='problem_check']
  for u in users:
    indicadores.loc[u,'n_attempted'] += 1
  # Percentaje of problems attempted (at least once)
  indicadores['per_attempted'] = [len(i)/len(prob_list) for i in nongraded['List of grades first attempt']]
  # Percentaje of problems correct on the first attempt (CFA)
  indicadores['CFA'] = [sum(i)/len(i) if len(i)!= 0 else '0 problems attempted' for i in nongraded['List of grades first attempt']]
  # Average grade of problems attempted (taking into account the maximum grade) -> ¿Tiene sentido si lo estamos haciendo por semanas?
  indicadores['avg_grade'] = [sum(i)/len(i) if len(i)!= 0 else '0 problems attempted' for i in nongraded['List of grades maximum grade']]

  # Average number of attempts (taking into account only attempted exercises)
  nongraded['List of attempts'] = nongraded['List of attempts'].fillna({i: [] for i in nongraded.index})
  for i in prob_list:
    y = df[(df['element_id'].str.contains(i)) & (df['event_type']=='problem_check')]
    y = y.sort_values('attempts')
    users_id = list(y['user_id'])
    attempts = list(y['attempts'])
    unique_users = np.unique(np.array(users_id))
    if len(users_id) == len(attempts):
        for u in unique_users:
          att = []
          for j in range(len(users_id)):
            if users_id[j] == u:
              att.append(attempts[j])
          nongraded['List of attempts'].loc[u].append(max(att))
  indicadores['avg_attempts'] = [sum(i)/len(i) if len(i)!= 0 else '0 problems attempted' for i in nongraded['List of attempts']]

  # Number of times the student asks for problem_show
  indicadores['n_show'] = 0
  y = df['user_id'].loc[(df['event_type']=='problem_show')]
  for user in y:
    indicadores.loc[user,'n_show'] += 1


  ### Videos
  print('Videos')
  # Percentaje of videos visualized
  vid_list = []
  for c in chaps:
    [vid_list.append(i) for i in videos['ID'].loc[videos['Chapter'].str.contains(str(c))]]
  play_video = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
  pause_stop_video = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
  for i in vid_list:
    play_video[i] = play_video[i].fillna({j: [] for j in play_video.index})
    pause_stop_video[i] = pause_stop_video[i].fillna({j: [] for j in pause_stop_video.index})
  for i in vid_list:
    # Play video
    users1 = list(df['user_id'].loc[(df['event_type']=='play_video') & (df['element_id'].str.contains(i))])
    seconds1 = list(df['old'].loc[(df['event_type']=='play_video') & (df['element_id'].str.contains(i))])
    for j in range(0,len(users1)):
      if not pd.isna(seconds1[j]):
        play_video.loc[users1[j],i].append(int(seconds1[j]))
    # Pause-stop video
    users2 = list(df['user_id'].loc[((df['event_type']=='stop_video') | (df['event_type']=='pause_video')) & (df['element_id'].str.contains(i))])
    seconds2 = list(df['old'].loc[((df['event_type']=='stop_video') | (df['event_type']=='pause_video')) & (df['element_id'].str.contains(i))])
    for j in range(0,len(users2)):
      if not pd.isna(seconds2[j]):
        pause_stop_video.loc[users2[j],i].append(int(seconds2[j]))
  videos_played = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
  videos_played = videos_played.T
  for i in vid_list:
    videos_played.loc[i] = list(videos['Seconds list'].loc[videos['ID']==i])
  videos_played = videos_played.T
  import ast
  for v in vid_list:
    for u in videos_played.index:
      play =  play_video.loc[u,v]
      pause = pause_stop_video.loc[u,v]
      if len(pause) > len(play):
        x = len(play)
        pause = pause[:len(play)]
      if len(play) > len(pause):
        x = len(pause)
        play = play[:len(pause)]
      if len(play) == len(pause):
        for i in range(0,len(play)):
          if pause[i] <= len(videos_played.loc[u,v]):
            if play[i] <= len(videos_played.loc[u,v]):
              if pause[i] > play[i]:
                videos_played.loc[u,v] = str(videos_played.loc[u,v])
                videos_played.loc[u,v] = ast.literal_eval(videos_played.loc[u,v].replace('.',','))
                if len(videos_played.loc[u,v][play[i]:pause[i]]) == pause[i]-play[i]:
                  videos_played.loc[u,v][play[i]:pause[i]] += np.ones(pause[i]-play[i])
  videos_percents = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
  for v in vid_list:
    videos_percents[v] = [np.sum(i)/len(i) for i in videos_played[v]]
  indicadores['per_videos'] = 0
  indicadores['per_videos'] = videos_percents.sum(axis=1)/len(vid_list)

  # Number of videos visualized (90% or more)
  indicadores['n_videos_watched'] = 0
  for u in indicadores.index:
    for v in videos_percents.columns:
      if videos_percents.loc[u,v] >= 0.9:
        indicadores.loc[u,'n_videos_watched'] += 1

  # Number of videos opened
  indicadores['n_videos_opened'] = 0
  print('maximum of: ', len(vid_list))
  for v in vid_list:
    vid_data = df['user_id'].loc[(df['element_id'].str.contains(v)) & (df['event_type'].str.contains('video'))]
    unique_users = vid_data.unique()
    for u in unique_users:
      indicadores.loc[u,'n_videos_opened'] += 1

  # Number of times the student pauses a video
  indicadores['n_pauses'] = 0
  users= df['user_id'].loc[(df['event_type']=='pause_video')]
  for u in users:
    indicadores.loc[u,'n_pauses'] += 1


  ### Forum
  print('Forum')
  # Number of threads created
  indicadores['n_threads'] = 0
  users = df['user_id'].loc[df['event_type']=='edx.forum.thread.created']
  for u in users:
    indicadores.loc[u,'n_threads'] += 1

  # Number of responses created
  indicadores['n_responses'] = 0
  users = df['user_id'].loc[df['event_type']=='edx.forum.response.created']
  for u in users:
    indicadores.loc[u,'n_responses'] += 1

  # Number of comments created
  indicadores['n_comments'] = 0
  users = df['user_id'].loc[df['event_type']=='edx.forum.comment.created']
  for u in users:
    indicadores.loc[u,'n_comments'] += 1


  ## Others
  print('Others')

  # Number of different days a student access the course
  indicadores['n_days'] = 0
  for d in days[w]:
    users_daily = df['user_id'].loc[df['time'].str.contains(d)]
    for u in set(list(users_daily)):
      indicadores.loc[u, 'n_days'] += 1

  if w == 1:
    indicadores.to_csv('./Modelos/Cummulative/Indicadores/indicadores_week{}.csv'.format('1_1'))
  else:
    indicadores.to_csv('./Modelos/Cummulative/Indicadores/indicadores_week{}.csv'.format(str(1)+'_'+str(w)))

  #return indicadores

In [ ]:
correspondence = {1:[1], 2:[1], 3:[1,2], 4:[1,2,3], 5:[1,2,3,4], 6:[1,2,3,4,5], 7:[1,2,3,4,5]}
for w in correspondence.keys():
  print('Week:',w)
  print('Chapters:',correspondence[w])
  indicadores_per_week_cummulative(w,correspondence[w],data,list_nongraded,days,videos)

In [ ]:
ind = indicadores_per_week_extra(1,1,data,days,videos)

In [ ]:
ind['n_videos_opened'].unique()

In [ ]:
indicadores = pd.read_csv('./Modelos/Weekly indicadores/indicadores_week1.csv',index_col=0)
indicadores

In [ ]:
vid_list = list(videos['ID'].loc[videos['Chapter'].str.contains(str(1))])
# Number of videos opened
indicadores['n_videos_opened'] = 0
for v in vid_list:
  vid_data = df['user_id'].loc[(df['event_type'].str.contains('video')) | (df['element_id'].str.contains(v))]
  unique_users = np.unique(np.array(vid_data))
  for u in unique_users:
    indicadores.loc[u,'n_videos_opened'] += 1

In [ ]:
indicadores

In [ ]:
indicadores['n_videos_opened'] = 0
for v in vid_list:
  users =
vid_data = df.loc[(df['event_type'].str.contains('video'))]
unique_users = np.unique(np.array(vid_data['user_id']))
for u in unique_users:
  indicadores.loc[u,'n_videos_opened'] = len(vid_data['element_id'].loc[vid_data['user_id']==u].unique())

In [ ]:
videos_percents

In [ ]:
w = 1
c = 1
df = pd.read_csv('./Modelos/Weekly datasets/week{}.csv'.format(w),index_col=0)
print('Videos')
# Percentaje of videos visualized
vid_list = list(videos['ID'].loc[videos['Chapter'].str.contains(str(c))])
play_video = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
pause_stop_video = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
for i in vid_list:
  play_video[i] = play_video[i].fillna({j: [] for j in play_video.index})
  pause_stop_video[i] = pause_stop_video[i].fillna({j: [] for j in pause_stop_video.index})

In [ ]:
for i in vid_list:
  # Play video
  print(i,'out of',len(vid_list))
  users1 = list(df['user_id'].loc[(df['event_type']=='play_video') & (df['element_id'].str.contains(i))])
  seconds1 = list(df['old'].loc[(df['event_type']=='play_video') & (df['element_id'].str.contains(i))])
  for j in range(0,len(users1)):
    if not pd.isna(seconds1[j]):
      play_video.loc[users1[j],i].append(int(seconds1[j]))
  # Pause-stop video
  users2 = list(df['user_id'].loc[((df['event_type']=='stop_video') | (df['event_type']=='pause_video')) & (df['element_id'].str.contains(i))])
  seconds2 = list(df['old'].loc[((df['event_type']=='stop_video') | (df['event_type']=='pause_video')) & (df['element_id'].str.contains(i))])
  for j in range(0,len(users2)):
    if not pd.isna(seconds2[j]):
      pause_stop_video.loc[users2[j],i].append(int(seconds2[j]))

In [ ]:
videos_played = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
videos_played = videos_played.T
for i in vid_list:
  videos_played.loc[i] = list(videos['Seconds list'].loc[videos['ID']==i])
videos_played = videos_played.T
import ast
for v in vid_list:
  print(v,'out of',len(vid_list))
  for u in videos_played.index:
    play =  play_video.loc[u,v]
    pause = pause_stop_video.loc[u,v]
    if len(pause) > len(play):
      x = len(play)
      pause = pause[:len(play)]
    if len(play) > len(pause):
      x = len(pause)
      play = play[:len(pause)]
    if len(play) == len(pause):
      for i in range(0,len(play)):
        if pause[i] <= len(videos_played.loc[u,v]):
          if play[i] <= len(videos_played.loc[u,v]):
            if pause[i] > play[i]:
              videos_played.loc[u,v] = str(videos_played.loc[u,v])
              videos_played.loc[u,v] = ast.literal_eval(videos_played.loc[u,v].replace('.',','))
              if len(videos_played.loc[u,v][play[i]:pause[i]]) == pause[i]-play[i]:
                videos_played.loc[u,v][play[i]:pause[i]] += np.ones(pause[i]-play[i])
videos_percents = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
for v in vid_list:
  videos_percents[v] = [np.sum(i)/len(i) for i in videos_played[v]]

In [ ]:
videos_percents

In [ ]:
n_vid = pd.DataFrame(index = data['user_id'].unique(), columns=['n_videos'])
n_vid

In [ ]:
n_vid['n_videos'] = 0
c = 0
for u in n_vid.index:
  if c%10000 == 0:
    print(c)
  for v in videos_percents.columns:
    if videos_percents.loc[u,v] >= 0.9:
      n_vid.loc[u,'n_videos'] += 1
  c += 1

In [ ]:
indicadores['n_comments'] = 0
users = df['user_id'].loc[df['event_type']=='edx.forum.comment.created']
for u in users:
  indicadores.loc[u,'n_comments'] += 1

In [ ]:
n_vid['n_videos'].unique()

In [ ]:
indicadores['n_videos_opened'] = 0
for v in vid_list:
  vid_data = df['user_id'].loc[(df['event_type'].str.contains('video')) | (df['element_id'].str.contains(v))]
  unique_users = np.unique(np.array(vid_data))
  for u in unique_users:
    indicadores[u,'n_videos_opened'] += 1

In [ ]:
vid_data['element_id'].unique()

In [ ]:
for u in users:
  indicadores.loc[u,'n_videos_opened'] += 1

In [ ]:
days

In [ ]:
# Number of different days a student access the course
indicadores['n_days'] = 0
for w in range(1,8):
  for d in days[w]:
    print(d)
    users_daily = df['user_id'].loc[df['time'].str.contains(d)]
    for u in set(list(users_daily)):
      indicadores.loc[u, 'n_days'] += 1

In [ ]:
indicadores.to_csv('./Modelos/indicadores_28_4_14_06.csv')

In [ ]:
week1 = pd.read_csv('./Modelos/week1.csv',index_col=0)
week1

In [ ]:
trial_28_4 = week1.loc[week1['time'].str.contains('2015-04-28')]

In [ ]:
trial_28_4

In [ ]:
trial_indicadores = indicadores_per_week(1,data,list_nongraded,days,videos)

In [ ]:
trial_indicadores

In [ ]:
trial_indicadores.loc[trial_indicadores['n_attempted']!=0]

In [ ]:
u = 'xxxxx'
len(trial_28_4.loc[(trial_28_4['user_id']==u)&(trial_28_4['event_type']=='problem_check')])

In [ ]:
trial_indicadores.loc['xxxxx']

In [ ]:
c = 1
for i in prob_list:
  print('problem {} out of 134'.format(c))
  y = trial_28_4.loc[(trial_28_4['element_id'].str.contains(i))&(trial_28_4['event_type']=='problem_check')]
  #print(y['grade'],y['attempts'])
  y = y.sort_values('attempts')
  users_id = list(y['user_id'])
  grades = list(y['grade'])
  max_grades = list(y['max_grade'])
  unique_users = np.unique(np.array(users_id))
  for u in unique_users:
    grs.loc[u,i] = []
  if len(users_id) == len(grades) == len(max_grades):
    for j in range(0,len(users_id)):
        # Per user, per problem: we have a list with all the grades per attempt. So we can extract the number of attempts and the maximum grade
        grs.loc[users_id[j],i].append(grades[j]/max_grades[j])
  c+=1

grs = grs.fillna('not attempted')
grs_one = grs.loc[(grs!='not attempted').any(axis=1)]

In [ ]:
nongraded = pd.DataFrame(index = data['user_id'].unique(), columns=['List of grades first attempt','List of grades maximum grade','Number of nongraded attempted (at least once)','Average grade of nongraded attempted','First attempt correct nongraded (%)','Average number of attempts'])
nongraded['List of grades maximum grade'] = nongraded['List of grades maximum grade'].fillna({i: [] for i in nongraded.index})
nongraded['List of grades first attempt'] = nongraded['List of grades first attempt'].fillna({i: [] for i in nongraded.index})
c = 1
users = grs_one.index
for i in prob_list:
  print('problem {} out of 134'.format(c))
  for u in users:
    if type(grs_one.loc[u,i]) == list:
      if len(grs_one.loc[u,i]) >= 1:
        # Max grade to get average
        nongraded.loc[u,'List of grades maximum grade'].append(grs_one.loc[u,i][-1])
        # Grade of first attempt
        if grs_one.loc[u,i][0] == 1:
          nongraded.loc[u,'List of grades first attempt'].append(1)
        else:
          nongraded.loc[u,'List of grades first attempt'].append(0)
  c += 1

In [ ]:
nongraded['Number of nongraded attempted (at least once)'] = [len(i) for i in nongraded['List of grades first attempt']]
#nongraded['Percentaje of nongraded attempted (at least once)'] = [i/134 for i in nongraded['Number of nongraded attempted (at least once)']]
nongraded['First attempt correct nongraded (%)'] = [sum(i)/len(i) if len(i)!= 0 else '0 problems attempted' for i in nongraded['List of grades first attempt']]
nongraded['Average grade of nongraded attempted'] = [sum(i)/len(i) if len(i)!= 0 else '0 problems attempted' for i in nongraded['List of grades maximum grade']]
nongraded

In [ ]:
nongraded['First attempt correct nongraded (%)'].loc[(nongraded['First attempt correct nongraded (%)'] != '0 problems attempted')]

In [ ]:
trial_28_4.loc[trial_28_4['user_id'] == 'xxxxx']

In [ ]:
nongraded['Average number of attempts'] = nongraded['Average number of attempts'].fillna({i: [] for i in nongraded.index})
c = 0
for i in prob_list:
  print('problem {} out of 134'.format(c))
  y = trial_28_4[(trial_28_4['element_id'].str.contains(i)) & (trial_28_4['event_type']=='problem_check')]
  y = y.sort_values('attempts')
  users_id = list(y['user_id'])
  attempts = list(y['attempts'])
  unique_users = np.unique(np.array(users_id))
  if len(users_id) == len(attempts):
      for u in unique_users:
        att = []
        for j in range(len(users_id)):
          if users_id[j] == u:
            att.append(attempts[j])
        nongraded['List of attempts'].loc[u].append(max(att))
  c += 1
nongraded['Average number of attempts'] = [sum(i)/len(i) if len(i)!= 0 else '0 problems attempted' for i in nongraded['List of attempts']]

In [ ]:
nongraded['nshow'] = 0
y = week1['user_id'].loc[(week1['event_type']=='problem_show')]
for row in y:
  nongraded.loc[row,'nshow'] += 1

In [ ]:
nongraded['n_pauses'] = 0
y = week1['user_id'].loc[(week1['event_type']=='pause_video')]
for row in y:
  nongraded.loc[row,'n_pauses'] += 1

In [ ]:
nongraded['n_threads'] = 0
users = week1['user_id'].loc[week1['event_type']=='edx.forum.thread.created']
for i in users:
  nongraded.loc[i,'n_threads'] += 1

In [ ]:
nongraded['n_comments'] = 0
users = week1['user_id'].loc[week1['event_type']=='edx.forum.comment.created']
for i in users:
  nongraded.loc[i,'n_comments'] += 1

In [ ]:
nongraded['n_responses'] = 0
users = week1['user_id'].loc[week1['event_type']=='edx.forum.response.created']
for i in users:
  nongraded.loc[i,'n_responses'] += 1

In [ ]:
for u in users:
  days = []
  all = data['time'].loc[(data['user_id']==u)]


In [ ]:
nongraded['n_threads']

In [ ]:
nongraded.loc[nongraded['n_searches']!=0]

In [ ]:
nongraded.loc['xxxxx']

In [ ]:
len(week1.loc[(week1['event_type'].str.contains('comment.created'))&(week1['user_id'] == 'xxxxx')])

In [ ]:
data['event_type'].loc[data['event_type'].str.contains('edx.forum')].unique()

In [ ]:
data.loc[data['event_type'] == 'edx.forum.searched']

In [ ]:
len(data['event_type'].loc[data['event_type'].str.contains('comment')])

In [ ]:
len(data['event_type'].loc[data['event_type'].str.contains('response')])

In [ ]:
for i in grs_one.loc['xxxxx']:
  if i != 'not attempted':
    print(i)

In [ ]:
nongraded.loc['xxxxx','Average number of attempts']

In [ ]:
week1['user_id'].loc[(week1['event_type']=='problem_show')]

In [ ]:
nongraded.loc['xxxxx','nshow']

In [ ]:
week1[(data['user_id']=='xxxxx')&(data['event_type']=='problem_show')]

In [ ]:
dict(nongraded['nshow'])

In [ ]:
y = week1.loc[(data['element_id'].str.contains('ef61adc076c140188b2f93f1080544e8'))&(week1['event_type']=='problem_check')]
print(y['grade'],y['attempts'])
y = y.sort_values('attempts')
print(y['grade'],y['attempts'])
users_id[-1]
users_id = list(y['user_id'])
grades = list(y['grade'])
max_grades = list(y['max_grade'])
y.loc[y['user_id'] == 'xxxxx']

In [ ]:
[print(i) for i in grs_one.loc['xxxxx'] if i!='not attempted']

In [ ]:
nongraded.loc[grs_one.index]

In [ ]:
y = week1[(data['element_id'].str.contains('xxxxx')) & (data['event_type']=='problem_check')]
y = y.sort_values('attempts')
print(y)

In [ ]:
for i in prob_list:
  print(week1.loc[(week1['event_type']=='problem_check') & (week1['element_id'].str.contains(i))])

In [ ]:
users_id = list(y['user_id'])
attempts = list(y['attempts'])

In [ ]:
attempts[3]

In [ ]:
[sum(i)/len(i) for i in nongraded['Average number of attempts']]

In [ ]:
nongraded['Average number of attempts'] = 0
unique_users = np.unique(np.array(users_id))
nongraded['Average number of attempts'] = nongraded['Average number of attempts'].fillna({i: [] for i in nongraded.index})
if len(users_id) == len(attempts):
    for u in unique_users:
      att = []
      for j in range(len(users_id)):
        if users_id[j] == u:
          att.append(attempts[j])
        #print(users_id[j])
        #print(u)
      print(att)

In [ ]:
nongraded['Average number of attempts']

In [ ]:
users_id = list(data['user_id'].loc[(data['event_type']=='problem_check') & (data['element_id'].str.contains(i))])
  attempts = list(data['attempts'].loc[(data['event_type']=='problem_check') & (data['element_id'].str.contains(i))])
  #unique_users = np.unique(np.array(users_id))
  if len(users_id) == len(attempts):
    #for u in unique_users:
    for j in range(0,len(users_id)):
      """print(users_id[j],i)
      print(c,attempts[j])"""
      if attempts[j] > indicadores1.loc[users_id[j]][1][c]:
        indicadores1.loc[users_id[j]][1][c] = attempts[j]
  c += 1

In [ ]:
#  - Average number of attempts (taking into account only attempted exercises)
#  - Number of times the student asks for problem_show

In [ ]:
y = data.loc[(data['element_id'].str.contains('xxxxx'))&(data['user_id']=='xxxxx')&(data['event_type']=='problem_check')]
print(y['grade'],y['attempts'])
y = y.sort_values('attempts')
print(y['grade'],y['attempts'])

In [ ]:
grs_one.loc['xxxxxx','xxxxxx']

In [ ]:
videos = pd.read_csv('./Indicadores/Datasets creados/videos.csv')
videos['Seconds list'] = [np.zeros((int(i))) for i in videos['Duration']]
videos

In [ ]:
v = pd.read_csv('./Indicadores/Datasets creados/videos_percents.csv')

In [ ]:
v

In [ ]:
data.loc[(data['user_id'].str.contains('xxxxx'))&(data['event_type'].str.contains('video'))]

In [ ]:
vid_list = videos['ID']


play_video = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
pause_stop_video = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
for i in vid_list:
  play_video[i] = play_video[i].fillna({j: [] for j in play_video.index})
  pause_stop_video[i] = pause_stop_video[i].fillna({j: [] for j in pause_stop_video.index})

for i in vid_list:
  # Play video
  users1 = list(trial_28_4['user_id'].loc[(trial_28_4['event_type']=='play_video') & (trial_28_4['element_id'].str.contains(i))])
  seconds1 = list(trial_28_4['old'].loc[(trial_28_4['event_type']=='play_video') & (trial_28_4['element_id'].str.contains(i))])
  for j in range(0,len(users1)):
    if not pd.isna(seconds1[j]):
      play_video.loc[users1[j],i].append(int(seconds1[j]))

  # Pause-stop video
  users2 = list(trial_28_4['user_id'].loc[((trial_28_4['event_type']=='stop_video') | (trial_28_4['event_type']=='pause_video')) & (trial_28_4['element_id'].str.contains(i))])
  seconds2 = list(trial_28_4['old'].loc[((trial_28_4['event_type']=='stop_video') | (trial_28_4['event_type']=='pause_video')) & (trial_28_4['element_id'].str.contains(i))])
  for j in range(0,len(users2)):
    if not pd.isna(seconds2[j]):
      pause_stop_video.loc[users2[j],i].append(int(seconds2[j]))


In [ ]:
videos_played = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
videos_played = videos_played.T
for i in vid_list:
  videos_played.loc[i] = list(videos['Seconds list'].loc[videos['ID']==i])
videos_played = videos_played.T

In [ ]:
for v in vid_list:
  print(v)
  for u in videos_played.index:
    play =  play_video.loc[u,v]
    pause = pause_stop_video.loc[u,v]
    if len(pause) > len(play):
      x = len(play)
      pause = pause[:len(play)]
    if len(play) > len(pause):
      x = len(pause)
      play = play[:len(pause)]
    if len(play) == len(pause):
      for i in range(0,len(play)):
        if pause[i] <= len(videos_played.loc[u,v]):
          if play[i] <= len(videos_played.loc[u,v]):
            if pause[i] > play[i]:
              if len(videos_played.loc[u,v][play[i]:pause[i]]) == pause[i]-play[i]:
                videos_played.loc[u,v][play[i]:pause[i]] += np.ones(pause[i]-play[i])
videos_percents = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
for v in vid_list:
  videos_percents[v] = [np.sum(i)/len(i) for i in videos_played[v]]

In [ ]:
videos_percents

In [ ]:

nongraded['ndays']

In [ ]:
i = 0
nongraded['ndays'] = 0
for u in unique_users:
  if i%1000 == 0:
    print(i)
  days = [t.split('T')[0] for t in week1['time'].loc[week1['user_id']==u]]
  nongraded.loc[u, 'ndays'] = len(set(days))
  i += 1

In [ ]:
week1

In [ ]:
data2015 = data.loc[data['time'].str.contains('2015-')]

import datetime

date_format = "%Y-%m-%d"
d1 = datetime.datetime.strptime("2015-04-28", date_format).date()
d2 = datetime.datetime.strptime("2015-06-30", date_format).date()
step = datetime.timedelta(days=7)
weeks = []
while d1 <= d2:
    #print(d1.strftime(date_format))
    weeks.append(d1.strftime(date_format))
    d1 += step

days = {}
date_format = "%Y-%m-%d"
for w in range(0,len(weeks[:-1])):
  days[w+1] = []
  #print('week:',w)
  d1 = datetime.datetime.strptime(weeks[w], date_format).date()
  d2 = datetime.datetime.strptime(weeks[w+1], date_format).date()
  step = datetime.timedelta(days=1)
  while d1 < d2:
      days[w+1].append(d1.strftime(date_format))
      d1 += step

print(days)

In [ ]:
week1

In [ ]:
week2 = pd.read_csv('./Modelos/week2.csv',index_col=0)
week2

In [ ]:
week3 = pd.read_csv('./Modelos/week3.csv',index_col=0)
week3

In [ ]:
i = 0
nongraded['ndays'] = 0
for d in days[1]:
  print(d)
  users_daily = week1['user_id'].loc[week1['time'].str.contains(d)]
  print(len(set(list(users_daily))))
  for u in set(list(users_daily)):
    nongraded.loc[u, 'ndays'] += 1

In [ ]:
users_daily

In [ ]:
data_v = set(list(week1['element_id'].loc[(week1['event_type']=='play_video')]))

In [ ]:
len(data_v)

In [ ]:
data_v2 = [i for i in set(list(week2['element_id'].loc[(week2['event_type']=='play_video')])) if i not in data_v]
len(data_v2)

In [ ]:
data_v3 = [i for i in set(list(week3['element_id'].loc[(week3['event_type']=='play_video')])) if i not in data_v and i not in data_v2]
len(data_v3)

In [ ]:
data_v4 = [i for i in set(list(week4['element_id'].loc[(week4['event_type']=='play_video')])) if i not in data_v and i not in data_v3]
len(data_v3¡4)

In [ ]:
list_nongraded = pd.read_csv('./Indicadores/list_nongraded.csv',index_col=0)
list_nongraded

In [ ]:
videos = pd.read_csv('./Indicadores/videos.csv')
videos['Seconds list'] = [np.zeros((int(i))) for i in videos['Duration']]
videos

In [ ]:
w = 1
list(videos.loc[videos['Chapter'].str.contains(str(w))].index)

In [ ]:
w = 1
vid_list = list(videos['ID'].loc[videos['Chapter'].str.contains(str(w))])

play_video = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
pause_stop_video = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
for i in vid_list:
  play_video[i] = play_video[i].fillna({j: [] for j in play_video.index})
  pause_stop_video[i] = pause_stop_video[i].fillna({j: [] for j in pause_stop_video.index})

for i in vid_list:
  # Play video
  users1 = list(trial_28_4['user_id'].loc[(trial_28_4['event_type']=='play_video') & (trial_28_4['element_id'].str.contains(i))])
  seconds1 = list(trial_28_4['old'].loc[(trial_28_4['event_type']=='play_video') & (trial_28_4['element_id'].str.contains(i))])
  for j in range(0,len(users1)):
    if not pd.isna(seconds1[j]):
      play_video.loc[users1[j],i].append(int(seconds1[j]))

  # Pause-stop video
  users2 = list(trial_28_4['user_id'].loc[((trial_28_4['event_type']=='stop_video') | (trial_28_4['event_type']=='pause_video')) & (trial_28_4['element_id'].str.contains(i))])
  seconds2 = list(trial_28_4['old'].loc[((trial_28_4['event_type']=='stop_video') | (trial_28_4['event_type']=='pause_video')) & (trial_28_4['element_id'].str.contains(i))])
  for j in range(0,len(users2)):
    if not pd.isna(seconds2[j]):
      pause_stop_video.loc[users2[j],i].append(int(seconds2[j]))

In [ ]:
videos_played = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
videos_played = videos_played.T
for i in vid_list:
  videos_played.loc[i] = list(videos['Seconds list'].loc[videos['ID']==i])
videos_played = videos_played.T

import ast

for v in vid_list:
  for u in videos_played.index:
    play =  play_video.loc[u,v]
    pause = pause_stop_video.loc[u,v]
    if len(pause) > len(play):
      x = len(play)
      pause = pause[:len(play)]
    if len(play) > len(pause):
      x = len(pause)
      play = play[:len(pause)]
    if len(play) == len(pause):
      for i in range(0,len(play)):
        if pause[i] <= len(videos_played.loc[u,v]):
          if play[i] <= len(videos_played.loc[u,v]):
            if pause[i] > play[i]:
              videos_played.loc[u,v] = str(videos_played.loc[u,v])
              videos_played.loc[u,v] = ast.literal_eval(videos_played.loc[u,v].replace('.',','))
              if len(videos_played.loc[u,v][play[i]:pause[i]]) == pause[i]-play[i]:
                videos_played.loc[u,v][play[i]:pause[i]] += np.ones(pause[i]-play[i])
videos_percents = pd.DataFrame(index = data['user_id'].unique(), columns=vid_list)
for v in vid_list:
  videos_percents[v] = [np.sum(i)/len(i) for i in videos_played[v]]

In [ ]:
videos_percents

In [ ]:
videos_played

In [ ]:
videos_percents['xxxxx'].loc[videos_percents['xxxxxx']!=0]

In [ ]:
perc_video =  pd.DataFrame(index = data['user_id'].unique(), columns=['perc_video'])
perc_video['perc_video'] = 0
perc_video['perc_video'] = videos_percents.sum(axis=1)/len(vid_list)

In [ ]:
perc_video['perc_video'] = videos_percents.sum(axis=1)/len(vid_list)

In [ ]:
perc_video.loc[perc_video['perc_video']!=0]

In [ ]:
videos_percents.loc['xxxxx']

In [ ]:
videos_percents.sum()

In [ ]:
w = 1
count_prob = pd.DataFrame(index = data['user_id'].unique(), columns=['count_prob'])
count_prob['count_prob'] = 0
prob_list = list(list_nongraded['ID'].loc[list_nongraded['Chapter'].str.contains(str(w))])
for p in prob_list:
    users = list(week1['user_id'] == week1.loc[(week1['element_id'].str.contains(p)) & (week1['event_type']=='problem_check')])
    for u in users:
      count_prob.loc[u,p] += 1

users = df['user_id'].loc[df['event_type']=='edx.forum.comment.created']
for u in users:
    indicadores.loc[u,'n_comments'] += 1